In [9]:
import pandas as pd
import numpy as np

In [6]:
from ultralytics import YOLO

test_model_path = 'models/best_player.pt'
test_vid = '/scratch/network/db0197/LabelingProcess/Squash Data/elias_makin_london2025/rallies/rally6_v2.mp4'

model = YOLO(test_model_path)

results = model.predict(
    source=test_vid,
    save_conf=True,
    project=f'rally6_v2/player_poses_predict', 
    save=True,
)


WARNING ⚠️ 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

video 1/1 (frame 1/1630) /scratch/network/db0197/LabelingProcess/Squash Data/elias_makin_london2025/rallies/rally6_v2.mp4: 384x640 4 persons, 9.4ms
video 1/1 (frame 2/1630) /scratch/network/db0197/LabelingProcess/Squash Data/elias_makin_london2025/rallies/rally6_v2.mp4: 384x640 4 persons, 9.0ms
video 1/1 (frame 3/1630) /scratch/network/db0197/LabelingProcess/Squash Data/elias_makin_london2025/rallies/rally6_v2.mp4: 384x640 4 persons, 9.0ms
video 1/1 (fra

In [15]:
import os

test_vid = '/scratch/network/db0197/LabelingProcess/Squash Data/elias_makin_london2025/rallies/rally6_v2.mp4'
print(os.path.exists(test_vid))  # check if this prints True

True


In [17]:
test_model_path = 'models/best_player.pt'
test_vid = '/scratch/network/db0197/LabelingProcess/Squash Data/elias_makin_london2025/rallies/rally6_v2.mp4'

model = YOLO(test_model_path)

results = model.track(
    source=test_vid,
    save_conf=True,
    project=f'rally6_v2/player_poses_track', 
    save=True,
    tracker="/scratch/network/db0197/Pipeline/models/custom_botsort.yaml"
)


WARNING ⚠️ 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

WARNING ⚠️ 'source' is missing. Using 'source=/scratch/network/db0197/.conda/envs/label/lib/python3.12/site-packages/ultralytics/assets'.
video 1/1 (frame 1/1630) /scratch/network/db0197/LabelingProcess/Squash Data/elias_makin_london2025/rallies/rally6_v2.mp4: 384x640 4 persons, 10.0ms
video 1/1 (frame 2/1630) /scratch/network/db0197/LabelingProcess/Squash Data/elias_makin_london2025/rallies/rally6_v2.mp4: 384x640 4 persons, 9.2ms
video 1/1 (frame 3/1630

In [24]:
res = list(results)

res[0].boxes

ultralytics.engine.results.Boxes object with attributes:

cls: tensor([0., 0., 0., 0.])
conf: tensor([0.9099, 0.8951, 0.5216, 0.3163])
data: tensor([[6.7643e+02, 6.5232e+02, 7.8622e+02, 9.2547e+02, 1.0000e+00, 9.0993e-01, 0.0000e+00],
        [1.2134e+03, 6.1257e+02, 1.3215e+03, 8.7493e+02, 2.0000e+00, 8.9508e-01, 0.0000e+00],
        [0.0000e+00, 4.4002e+02, 8.7300e+01, 6.4931e+02, 3.0000e+00, 5.2156e-01, 0.0000e+00],
        [1.6865e+03, 4.8019e+02, 1.8102e+03, 6.4006e+02, 4.0000e+00, 3.1634e-01, 0.0000e+00]])
id: tensor([1., 2., 3., 4.])
is_track: True
orig_shape: (1080, 1920)
shape: torch.Size([4, 7])
xywh: tensor([[ 731.3274,  788.8956,  109.7878,  273.1428],
        [1267.4507,  743.7463,  108.1958,  262.3613],
        [  43.6500,  544.6679,   87.3000,  209.2859],
        [1748.3336,  560.1270,  123.7117,  159.8711]])
xywhn: tensor([[0.3809, 0.7305, 0.0572, 0.2529],
        [0.6601, 0.6887, 0.0564, 0.2429],
        [0.0227, 0.5043, 0.0455, 0.1938],
        [0.9106, 0.5186, 0.0644

In [12]:
KEYPOINT_NAMES = ['front_left', 'mid_left', 'mid_right', 'front_right', 'service_right', 'service_left']
KP_COLS = [f'{kp}_{s}' for kp in KEYPOINT_NAMES for s in ('x', 'y', 'vis')]
COLS = ['frame'] + KP_COLS

def parse_gt_txt(txt_path, save_path, img_w, img_h):
    """Throwaway: converts a YOLO keypoint label txt to a CSV matching court_keypoints.csv format.

    The txt format is:
        class_id cx cy w h kp1_x kp1_y kp1_vis ... kp6_x kp6_y kp6_vis  (normalized 0-1)

    Assumes a single-line file (one ground-truth annotation).
    Sets vis=1.0 for all keypoints so get_avg_keypoints() keeps them all.
    """
    with open(txt_path) as f:
        vals = list(map(float, f.readline().split()))

    # skip: class_id(1) + bbox cx,cy,w,h(4) = first 5 values
    kp_arr = np.array(vals[5:]).reshape(6, 3)  # (6, 3): x, y, vis
    kp_arr[:, 0] *= img_w
    kp_arr[:, 1] *= img_h
    # kp_arr[:, 2] = 1.0  # set vis=1.0 so get_avg_keypoints keeps this row

    flat = kp_arr.ravel()
    df = pd.DataFrame([np.concatenate([[0], flat])], columns=COLS)
    df['frame'] = df['frame'].astype(int)
    df.to_csv(save_path, index=False)
    return df

parse_gt_txt(
    'data/rally6_v2/court_labels/makin_ground_truth.txt',
    save_path='data/rally6_v2/court_labels/makin_ground_truth.csv',
    img_w=1920, img_h=1080
)

,frame,front_left_x,front_left_y,front_left_vis,mid_left_x,mid_left_y,mid_left_vis,mid_right_x,mid_right_y,mid_right_vis,front_right_x,front_right_y,front_right_vis,service_right_x,service_right_y,service_right_vis,service_left_x,service_left_y,service_left_vis
0,0,565.867,554.7579,2.0,484.333,812.4311,2.0,1432.6566,810.2588,2.0,1352.1416,552.5554,2.0,1359.1234,338.9695,2.0,555.9963,339.5956,2.0


In [15]:
from visualizer import visualize_court_points
data_path = 'data/rally6_v2/court_labels/makin_ground_truth.csv'

from calibration import get_avg_keypoints
gt_avg = get_avg_keypoints(data_path, min_vis=0)
gt_avg.values

def pairwise(iterable):
        it = iter(iterable)
        return list(zip(it, it))

gt_points = pairwise(gt_avg.values)

Before filtering: (1, 19)
After filtering: (1, 19)


[(np.float64(565.867), np.float64(554.7579)),
 (np.float64(484.3330000000001), np.float64(812.4311)),
 (np.float64(1432.6566), np.float64(810.2588)),
 (np.float64(1352.1416), np.float64(552.5554)),
 (np.float64(1359.1234), np.float64(338.9695)),
 (np.float64(555.9963), np.float64(339.5956))]

In [1]:
import cv2

path = '/scratch/network/db0197/LabelingProcess/Squash Data/elias_makin_london2025/rallies/rally6_v2.mp4' 
cap = cv2.VideoCapture(path)
success = True
i = 0
while success:
    success,image = cap.read()
    if success:
        cv2.imwrite(f'/scratch/network/db0197/Pipeline/data/rally6_v2/frames/frame{i:04d}.jpg', image)
        i+=1

In [2]:
cap.get(cv2.CAP_PROP_FPS)

50.0

In [38]:
import json
import numpy as np
import pandas as pd
from scipy.optimize import minimize, least_squares
import calibration

def create_3d_trajectory(params, times):
    '''
    Creates a trajectory of 3D world points given initial conditions. 

    ** Currently Implemented with a basic newton + drag equation **

    Args:
        params: The intial position (x,y,z) and initial velocity (vx0, vy0, vz0) of the ball. Each is an element of a list. 

    Returns:
        Times x 4 list of 3D homogenous coords in real world space based on intial conditions.
    '''

    x0, y0, z0, vx0, vy0, vz0, drag = params
    g = 9.81

    n_points = len(times)
    positions = np.zeros((n_points, 4))

    for i, t in enumerate(times):
        positions[i, 0] = x0 + vx0 * t * np.exp(-drag * t)
        positions[i, 1] = y0 + vy0 * t * np.exp(-drag * t)
        positions[i, 2] = z0 + vz0 * t - 0.5 * g * t ** 2
        positions[i, 3] = 1 # homogenous coords

    return positions

def penalize_out_of_bounds(points_3d):
    '''
    Check if 3D position is within squash court bounds.

    Coordinate system:
    x: width, -3.2 (left wall) to 3.2 (right wall)
    y: depth,  -4.31 (back wall) to 5.44 (front wall), with 0 at center of T
    z: height, 0 (floor) to 10m max

    Args:
        points_3d: a homogenous 3D coordinate

    Returns:
        penalty: the penalty to be applied proportional to how far out of court bounds the ball is.
    '''
    x, y, z, _ = points_3d
    penalty = 0.0

    if x < -3.2 or x > 3.2:
        penalty += 1000 * abs(min(x + 3.2, 0) + max(x - 3.2, 0))

    if y < -4.31 or y > 5.44:
        penalty += 1000 * abs(min(y + 4.31, 0) + max(y - 5.44, 0))

    if z < 0 or z > 10.0:
        penalty += 1000 * abs(min(z, 0) + max(z - 10.0, 0))

    return penalty

def relu(x):
    return np.maximum(0.0, x)

def court_violation_residuals(points_3d, weight=10.0):
    x_min, x_max = -3.2, 3.2
    y_min, y_max = -4.31, 5.44
    z_min, z_max = 0.0, 10.0

    x = points_3d[:, 0]
    y = points_3d[:, 1]
    z = points_3d[:, 2]

    vx_hi = relu(x - x_max)
    vx_lo = relu(x_min - x)
    vy_hi = relu(y - y_max)
    vy_lo = relu(y_min - y)
    vz_hi = relu(z - z_max)
    vz_lo = relu(z_min - z)

    w = np.sqrt(weight)
    return w * np.concatenate([vx_hi, vx_lo, vy_hi, vy_lo, vz_hi, vz_lo], axis=0)

def compute_loss(points_2d, projected_points):
    '''MSE reprojection error between ground truth and projected 2D points.'''
    diff = points_2d - projected_points
    return np.mean(diff ** 2)

def predict_3d_trajectory(seg_points_2d, start, stop, P, FPS):
    times = np.arange(stop - start + 1, dtype=np.float64) / FPS
    detected_points = seg_points_2d.values.astype(np.float64)  # Nx2

    def residuals(params):
        points_3d = create_3d_trajectory(params, times)              # Nx4
        proj_points = calibration.project_points(P, points_3d)       # Nx2

        reproj_res = (proj_points - detected_points).reshape(-1)     # (2N,)

        court_res = court_violation_residuals(points_3d, weight=10.0) # tune weight if needed
        return np.concatenate([reproj_res, court_res], axis=0)

    x0 = np.array([0.0, 0.0, 1.0,
                   0.0, 0.0, 0.0,
                   0.1], dtype=np.float64)

    lb = np.array([-3.2, -4.31, 0.0,
                   -30.0, -30.0, -20.0,
                   0.0], dtype=np.float64)
    ub = np.array([ 3.2,  5.44, 10.0,
                    30.0,  30.0,  20.0,
                    5.0], dtype=np.float64)

    result = least_squares(
        residuals,
        x0,
        bounds=(lb, ub),
        method="trf",
        loss="soft_l1",
        f_scale=3.0,
        x_scale="jac",
        ftol=1e-6,
        xtol=1e-6,
        gtol=1e-6,
        max_nfev=20000
    )

    return create_3d_trajectory(result.x, times)

def predict_3d_trajectory_old(seg_points_2d, start, stop, P, FPS):
    '''
    Optimizes 3D trajectory params to minimize reprojection error against 2D (u, v) ground truth.
    Args: 
        seg_points_2d: Nx2 DataFrame of (u, v) pixel coordinates for this segment
        P: 3x4 projection matrix from calibration
    Returns: 
        Nx3 array of optimized 3D positions
    '''
    times = np.arange(stop - start + 1) / FPS
    detected_points = seg_points_2d.values  # Nx2 numpy array

    def objective(params):
        points_3d = create_3d_trajectory(params, times)
        proj_points = calibration.project_points(P, points_3d)
        loss = compute_loss(detected_points, proj_points)
        for point in points_3d:
            loss += penalize_out_of_bounds(point)
        return loss

    # Initial guess: center of T, 1m height, zero velocity, low drag
    x0 = [0.0, 0.0, 1.0,
          0.0, 0.0, 0.0,
          0.1]

    result = minimize(objective, x0, method='Nelder-Mead',
                      options={'maxiter': 10000, 'xatol': 1e-4, 'fatol': 1e-4})

    return create_3d_trajectory(result.x, times)

def predict_segments(detected_points, segments, P, FPS):
    '''
    Predicts the 3D points that best optimize the loss function per segment. 
    Args:
        detected_points: A dataframe containing the X and Y detected ball points in (u, v) coordinates. our "ground truth"
        segments: A list of tuples of the form (start frame, end frame) for the individual segments
        P: the projection matrix that takes points from 3D to 2D
        FPS: The frames per second of the video being detected

    Returns:
        a list of trajectories, which are themselves list of 3D homogenous points in meters, with (0,0,0,1) being the T. 
    '''
    trajectories = []
    for start, stop in segments:
        seg_points_2d = detected_points.iloc[start:stop + 1]
        traj = predict_3d_trajectory(seg_points_2d, start, stop, P, FPS)
        trajectories.append(traj)
        proj_traj = calibration.project_points(P, traj)
        loss = compute_loss(seg_points_2d.values, proj_traj)
        print(f"Segment {start}-{stop}: {len(traj)} points, first={traj[0]}, last={traj[-1]}\nLoss: {loss}")

    return trajectories

In [43]:
test_court_csv = '/scratch/network/db0197/Pipeline/predictions/rally6_v2/court_keypoints.csv'
test_ball_csv = '/scratch/network/db0197/Pipeline/data/rally6_v2/ball_labels/rally6_v2_ball.csv'
test_segments_json = '/scratch/network/db0197/Pipeline/data/rally6_v2/ball_labels/manual_segment.json'
FPS = 50

detected_points = pd.read_csv(test_ball_csv)
detected_points = detected_points[['X', 'Y']]

avg = calibration.get_avg_keypoints(test_court_csv)
P = calibration.map_3dcoords(avg)

with open(test_segments_json, 'r') as file:
    data = json.load(file)
segments = [(seg['start_frame'], seg['end_frame']) for seg in data['segments']]

trajectories = predict_segments(detected_points, segments, P, FPS)

Before filtering: (1630, 19)
After filtering: (1630, 19)
Segment 43-54: 12 points, first=[ 0.12829451 -4.31        1.19553126  1.        ], last=[-0.96466454 -1.94087105  2.71928093  1.        ]
Loss: 25.186116435544907
Segment 54-94: 41 points, first=[-1.13915776  5.44        1.62525551  1.        ], last=[-4.079753    5.93921343 -0.98033626  1.        ]
Loss: 26.208267947199285
Segment 94-107: 14 points, first=[-2.74496957 -4.30999957  1.30923393  1.        ], last=[-2.89921681  1.46259203  2.22161727  1.        ]
Loss: 8.806651525696186
Segment 107-147: 41 points, first=[-2.62893379 -1.22557809  2.47692102  1.        ], last=[-3.92264309  4.36581889 -1.62972269  1.        ]
Loss: 13.57858390891895
Segment 147-174: 28 points, first=[-3.10071453e+00 -1.65500079e+00  4.41319368e-13  1.00000000e+00], last=[-2.94197647 -4.88850239  0.56062407  1.        ]
Loss: 4.030289595438547


In [42]:
from pathlib import Path
import visualizer

vid_path = Path('/scratch/network/db0197/LabelingProcess/Squash Data/elias_makin_london2025/rallies/rally6_v2.mp4')

visualizer.visualize_trajectories_on_video(
    trajectories, segments, P, vid_path, out_path='predictions/rally6_v2/annotated.mp4')

Saved annotated video to predictions/rally6_v2/annotated.mp4 (1630 frames)
